# 01 · Data exploration

This notebook looks at the three cleaned datasets the models learn from.

**Before running:** download the datasets and run `retrain_models.bat` (or `python -m services.data_preparation` inside `backend/`) so the files in `data/processed/` exist. Then run all cells from top to bottom.

In [ ]:
import sys
from pathlib import Path

# Jupyter starts inside notebooks/, so the project root is one level up.
PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "backend"))

import matplotlib.pyplot as plt
import pandas as pd

from services.skills import extract_skills

PROCESSED = PROJECT_ROOT / "data" / "processed"
plt.rcParams["figure.figsize"] = (10, 4)

## Dataset 1 — Resumes by job category (role classifier)

Each resume has one of 24 job categories. A balanced dataset (similar bar lengths) is easier to learn from.

In [ ]:
resumes = pd.read_csv(PROCESSED / "resumes_cleaned.csv")
print(f"{len(resumes)} resumes, {resumes['category'].nunique()} categories")

resumes["category"].value_counts().sort_values().plot.barh(figsize=(8, 7), title="Resumes per category")
plt.xlabel("Number of resumes")
plt.show()

How long are the resumes? Very short resumes give the model less to learn from.

In [ ]:
resumes["word_count"] = resumes["resume_text"].str.split().str.len()
resumes["word_count"].plot.hist(bins=50, title="Words per resume")
plt.xlabel("Words")
plt.show()

resumes["word_count"].describe()

## Dataset 2 — Resume–job pairs (matching model)

English pairs labelled *Good Fit*, *Potential Fit* or *No Fit*. Good and Potential Fit are treated as a **match** (worth interviewing).

In [ ]:
matching = pd.read_csv(PROCESSED / "matching_cleaned.csv")
print(f"{len(matching)} resume-job pairs")
matching["raw_status"].value_counts()

In [ ]:
matching["match_label"].map({1: "Match", 0: "No Fit"}).value_counts().plot.bar(title="Label balance", rot=0)
plt.ylabel("Pairs")
plt.show()

## Dataset 3 — Candidate profiles with recruiter decisions (screening models)

In [ ]:
screening = pd.read_csv(PROCESSED / "screening_cleaned.csv")
print(f"{len(screening)} candidates")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
screening["decision"].value_counts().plot.bar(ax=axes[0], title="Recruiter decision", rot=0)
screening["job_role"].value_counts().plot.bar(ax=axes[1], title="Job role", rot=20)
screening["ai_score"].plot.hist(ax=axes[2], bins=30, title="AI score (0-100)")
plt.tight_layout()
plt.show()

**AI score vs decision.** If the two boxes barely overlap, the decision is easy to predict from the score.
That is why the screening model reaches very high accuracy — worth saying honestly in the viva: this dataset looks computer-generated, so real-world accuracy would be lower.

In [ ]:
screening.boxplot(column="ai_score", by="decision", figsize=(6, 4))
plt.suptitle("")
plt.title("AI score by recruiter decision")
plt.show()

## Most common skills in the resumes (skill engine)

In [ ]:
skill_counts = pd.Series([skill for text in resumes["resume_text"] for skill in extract_skills(text)]).value_counts()

skill_counts.head(20).sort_values().plot.barh(figsize=(8, 6), title="Top 20 skills found in dataset 1")
plt.xlabel("Resumes mentioning the skill")
plt.show()